# POULTRYSENSE - YOLOv8 Training Notebook (Colab T4)

**Purpose:** Inspect the 3 datasets, spot-check behavior labels, then train 3 models:
- **Model A (Detection/Segmentation):** `yolov8n-seg` on Dataset1 (retrain in Part H2/H3 with fixed recipe)
- **Model C (Health):** classification on Dataset2 (Healthy / Unhealthy)
- **Model B (Behavior):** classification on Dataset3 crop-based labels (stand / peck / walk)

**Workflow:** Run cells **in order**. A STOP gate appears after the behavior label spot-check - review it and paste results back before training.

**Datasets must be at:** `/content/drive/MyDrive/PoultrySense/`

## PART A - Environment Check
Verify Python, PyTorch, CUDA, GPU, Ultralytics, storage before doing anything.

In [ ]:
import sys, platform, shutil, subprocess, os, torch

print('Python   :', sys.version.split()[0])
print('PyTorch  :', torch.__version__)
print('CUDA avail:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU      :', torch.cuda.get_device_name(0))

try:
    import ultralytics
    print('Ultralytics:', ultralytics.__version__)
except ImportError:
    print('Ultralytics NOT installed - installing...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    import ultralytics
    print('Ultralytics:', ultralytics.__version__)

import cv2, numpy, pandas, matplotlib
print('OpenCV   :', cv2.__version__)
print('NumPy    :', numpy.__version__)
print('Pandas   :', pandas.__version__)

total, used, free = shutil.disk_usage('/content')
print('Disk free:', round(free/1e9, 1), 'GB')

try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
except Exception as e:
    print('nvidia-smi unavailable:', e)

## PART B - Mount Drive and Locate Datasets
The 3 ZIPs must be in `/content/drive/MyDrive/PoultrySense/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/PoultrySense'
print('PoultrySense exists:', os.path.exists(BASE))
if os.path.exists(BASE):
    for f in os.listdir(BASE):
        full = os.path.join(BASE, f)
        if os.path.isfile(full):
            print(f'{f:40s} {os.path.getsize(full)/1e6:9.1f} MB')
        else:
            print(f'{f:40s} <dir>')

## PART C - Extract ZIPs (read-only vs originals)
Extracts into `/content/datasets/`. The original ZIPs on Drive are never modified.

In [ ]:
import zipfile, os

SRC = '/content/drive/MyDrive/PoultrySense'
DST = '/content/datasets'
os.makedirs(DST, exist_ok=True)

zips = {
    'D1_Detection': 'Dataset1_Detection.zip',
    'D2_Health': 'Dataset2_Health.zip',
    'D3_Behavior': 'Dataset3_Behavior.zip',
}

for name, fn in zips.items():
    zp = os.path.join(SRC, fn)
    out = os.path.join(DST, name)
    assert os.path.exists(zp), f'MISSING: {zp}'
    if not os.path.exists(out):
        print(f'Extracting {fn} ...')
        with zipfile.ZipFile(zp) as z:
            z.extractall(out)
        print('  done ->', out)
    else:
        print(f'Already extracted: {out}')

## PART D - Dataset1 (Detection/Segmentation) - Validate
Confirmed structure: YOLOv8 **segmentation** polygons, 1 class `chicken`, 106 images (train/valid/test). Verify all labels are valid YOLO-seg polygons and show samples.

In [ ]:
import os, glob, random

D1 = '/content/datasets/D1_Detection'
print('data.yaml:')
print(open(os.path.join(D1, 'data.yaml')).read())

for split in ['train', 'valid', 'test']:
    imgs = glob.glob(f'{D1}/{split}/images/*.jpg')
    labs = glob.glob(f'{D1}/{split}/labels/*.txt')
    empty = sum(1 for f in labs if os.path.getsize(f) == 0)
    print(f'{split:6s}: images={len(imgs):3d} labels={len(labs):3d} empty_labels={empty}')

# validate label files are well-formed YOLO-seg polygons
random.seed(0)
bad = 0
sample = random.choice(glob.glob(f'{D1}/train/labels/*.txt'))
print('sample label (train):')
print(open(sample).read()[:400])
for f in glob.glob(f'{D1}/train/labels/*.txt') + glob.glob(f'{D1}/valid/labels/*.txt'):
    for line in open(f):
        parts = line.split()
        if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
            bad += 1
            print('BAD LINE', f, line[:60])
print('malformed label lines:', bad)

Show sample segmentation images with polygon overlays drawn from the labels.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

def draw_seg(img_path, label_path):
    img = cv2.imread(img_path)
    H, W = img.shape[:2]
    overlay = img.copy()
    for line in open(label_path):
        parts = line.split()
        pts = np.array([[float(parts[i])*W, float(parts[i+1])*H] for i in range(1, len(parts), 2)], np.int32)
        cv2.fillPoly(overlay, [pts], (0, 200, 0))
    cv2.addWeighted(overlay, 0.4, img, 0.6, 0, img)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

imgs = sorted(glob.glob(f'{D1}/train/images/*.jpg'))[:4]
fig, axes = plt.subplots(1, len(imgs), figsize=(16, 4))
for ax, im in zip(axes, imgs):
    lab = im.replace('/images/', '/labels/').replace('.jpg', '.txt')
    ax.imshow(draw_seg(im, lab))
    ax.axis('off')
plt.show()

## PART E - Dataset2 (Health) - Prepare Classification
Confirmed: Roboflow **classification** dataset, classes **Healthy / Unhealthy**, 154 images (train/valid/test), labels in `_classes.csv`. Build YOLO-classify folder structure (train/ + val/ with class subfolders).

In [ ]:
import os, pandas as pd, shutil, glob

D2 = '/content/datasets/D2_Health'
CLS = '/content/health_cls'
if os.path.exists(CLS):
    shutil.rmtree(CLS)

for split in ['train', 'valid', 'test']:
    csvp = glob.glob(f'{D2}/{split}/_classes.csv')
    if not csvp:
        continue
    df = pd.read_csv(csvp[0])
    n_h = 0
    n_u = 0
    for _, r in df.iterrows():
        cls = 'Healthy' if r['Healthy'] == 1 else 'Unhealthy'
        if cls == 'Healthy': n_h += 1
        else: n_u += 1
        src = os.path.join(D2, split, r['filename'])
        dst = os.path.join(CLS, split, cls, r['filename'])
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(src, dst)
    print(f'{split:6s}: Healthy={n_h:3d} Unhealthy={n_u:3d} total={n_h+n_u}')

# distribution report
print()
print('Per-split image counts:')
for split in ['train', 'valid', 'test']:
    base = os.path.join(CLS, split)
    if os.path.exists(base):
        for cls in sorted(os.listdir(base)):
            n = len(os.listdir(os.path.join(base, cls)))
            print(f'  {split}/{cls}: {n}')

Show sample Healthy vs Unhealthy images.

In [ ]:
import cv2, matplotlib.pyplot as plt, numpy as np

healthy = sorted(glob.glob('/content/health_cls/train/Healthy/*.jpg'))[:3]
unhealthy = sorted(glob.glob('/content/health_cls/train/Unhealthy/*.jpg'))[:3]
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, im in enumerate(healthy + unhealthy):
    ax = axes[i // 3][i % 3]
    ax.imshow(cv2.cvtColor(cv2.imread(im), cv2.COLOR_BGR2RGB))
    ax.axis('off')
    ax.set_title('Healthy' if i < 3 else 'Unhealthy')
plt.show()

## PART F - Dataset3 (Behavior) - Extract Labeled Crops
Confirmed: 6 videos (1920x1080, ~400 frames each), per-frame JSON with **tracked bounding boxes** + **per-track actions (stand/peck/walk)**. Build a classification dataset by cropping each tracked chicken and labeling it with its action.

**Split is by VIDEO (no leakage):** train=000004/000007/000010, val=000008/000009, test=000011.

**Crops are padded** (40% margin + minimum 160px window) so chickens are NOT cut off, and a wider **context view** is saved alongside for label review.

In [ ]:
import os, json, cv2, glob, shutil

D3 = '/content/datasets/D3_Behavior/poultry_behaviour_dataset/chickens'
OUT = '/content/behavior_cls'
CTX = '/content/behavior_ctx'
for d in [OUT, CTX]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

VIDEO_SPLIT = {'000004': 'train', '000007': 'train', '000010': 'train',
               '000008': 'val',   '000009': 'val',
               '000011': 'test'}

rows = []
for jobdir in sorted(glob.glob(f'{D3}/job_*')):
    job = os.path.basename(jobdir)
    vid_id = job.split('_')[-1]
    ann = os.path.join(jobdir, f'annotations_{vid_id}.json')
    if not os.path.exists(ann):
        continue
    vid = os.path.join(jobdir, f'{job}_400frames.mp4')
    if not os.path.exists(vid):
        vid = os.path.join(jobdir, f'{job}.mp4')
    d = json.load(open(ann))
    cap = cv2.VideoCapture(vid)
    split = VIDEO_SPLIT[vid_id]
    frame_map = {f['frame_index']: f for f in d['frames']}
    need = sorted(frame_map.keys())
    saved = 0
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i in frame_map:
            f = frame_map[i]
            H, W = frame.shape[:2]
            acts = f.get('actions') or {}
            tb = f.get('tracks_bbox') or []
            for box in tb:
                x1, y1, x2, y2, tid, conf = box
                act = acts.get(str(int(tid)))
                if not act:
                    continue
                x1, x2 = max(0, int(x1)), min(W, int(x2))
                y1, y2 = max(0, int(y1)), min(H, int(y2))
                bw, bh = x2 - x1, y2 - y1
                if bw < 15 or bh < 15:
                    continue
                cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
                pad_w = int(bw * 0.4)
                pad_h = int(bh * 0.4)
                x1 = max(0, x1 - pad_w); x2 = min(W, x2 + pad_w)
                y1 = max(0, y1 - pad_h); y2 = min(H, y2 + pad_h)
                MIN_SIDE = 160
                half_w = max((x2 - x1) / 2.0, MIN_SIDE / 2.0)
                half_h = max((y2 - y1) / 2.0, MIN_SIDE / 2.0)
                x1 = int(max(0, cx - half_w)); x2 = int(min(W, cx + half_w))
                y1 = int(max(0, cy - half_h)); y2 = int(min(H, cy + half_h))
                if x2 - x1 < 40 or y2 - y1 < 40:
                    continue
                crop = frame[y1:y2, x1:x2]
                cls_dir = os.path.join(OUT, split, act)
                os.makedirs(cls_dir, exist_ok=True)
                cv2.imwrite(os.path.join(cls_dir, f'{vid_id}_t{int(tid)}_f{i:04d}.jpg'), crop)
                ctx_w = int((x2 - x1) * 2.2)
                ctx_h = int((y2 - y1) * 2.2)
                ctx_x1 = int(max(0, cx - ctx_w / 2)); ctx_x2 = int(min(W, cx + ctx_w / 2))
                ctx_y1 = int(max(0, cy - ctx_h / 2)); ctx_y2 = int(min(H, cy + ctx_h / 2))
                if ctx_x2 - ctx_x1 >= 40 and ctx_y2 - ctx_y1 >= 40:
                    ctx = frame[ctx_y1:ctx_y2, ctx_x1:ctx_x2]
                    ctx_dir = os.path.join(CTX, split, act)
                    os.makedirs(ctx_dir, exist_ok=True)
                    cv2.imwrite(os.path.join(ctx_dir, f'{vid_id}_t{int(tid)}_f{i:04d}.jpg'), ctx)
                rows.append((vid_id, act))
                saved += 1
        i += 1
        if i > max(need):
            break
    cap.release()
    print(f'{job}: saved {saved} labeled crops')

df = pd.DataFrame(rows, columns=['video', 'action'])
print()
print('Per-video action counts:')
print(df.groupby(['video', 'action']).size().unstack(fill_value=0))
print()
print('Total labeled crops by action:')
print(df['action'].value_counts())
print()
print('Total labeled crops:', len(df))

## PART G - D3 Behavior Label Verification  (REVIEW BEFORE TRAINING)
Three-step check. The stand/peck/walk labels were **auto-generated**, so we verify them 3 ways:
1. **Quantitative motion check** - objective measurement from the tracking data (does `walk` actually move more than `stand`?).
2. **Inline annotated videos** - watch each video with boxes+labels drawn.
3. **Padded crop grid** - close-up look at each class.

If the numbers and videos look right, the labels are usable. If `walk` is clearly wrong, tell us before training.

In [ ]:
import os, zipfile, json, math, statistics, collections

BASE = '/content/drive/MyDrive/PoultrySense'

def center(b):
    return ((b[0] + b[2]) / 2.0, (b[1] + b[3]) / 2.0)

D3_ZIP = '/content/datasets/D3_Behavior/poultry_behaviour_dataset/chickens'
stats = collections.defaultdict(list)
for n in ['000004', '000007', '000008', '000009', '000010', '000011']:
    ann = f'{D3_ZIP}/job_{n}/annotations_{n}.json'
    if not os.path.exists(ann):
        continue
    d = json.load(open(ann))
    prev = {}
    for f in d['frames']:
        acts = f.get('actions') or {}
        tb = f.get('tracks_bbox') or []
        for box in tb:
            x1, y1, x2, y2, tid, conf = box
            act = acts.get(str(int(tid)))
            if act is None:
                continue
            c = center(box)
            if tid in prev:
                spd = math.dist(c, prev[tid])
                stats[act].append(spd)
            prev[tid] = c

print('Quantitative motion check (px per frame):')
print('%-8s %10s %10s %10s %12s' % ('label', 'N', 'mean', 'median', 'pct moving>3'))
for act in ['stand', 'peck', 'walk']:
    s = stats.get(act, [])
    if not s:
        continue
    mean = statistics.mean(s)
    med = statistics.median(s)
    pct = 100.0 * sum(1 for v in s if v > 3.0) / len(s)
    print('%-8s %10d %10.3f %10.3f %11.1f%%' % (act, len(s), mean, med, pct))
print()
print('Interpretation: walk should show clearly MORE motion than stand/peck.')

In [ ]:
from IPython.display import Video, display

D3 = '/content/datasets/D3_Behavior/poultry_behaviour_dataset/chickens'
vids = ['000004', '000007', '000008', '000009', '000010', '000011']
for n in vids:
    v = f'{D3}/job_{n}/annotated_{n}.mp4'
    if os.path.exists(v):
        display(Video(v, embed=True, width=640))
    else:
        print('missing annotated video:', v)

In [ ]:
import cv2, glob, os, random, matplotlib.pyplot as plt, numpy as np

random.seed(0)
OUT = '/content/behavior_cls'
CTX = '/content/behavior_ctx'
N = 6

colors = {'stand': 'tab:red', 'peck': 'tab:blue', 'walk': 'tab:green'}
classes = ['stand', 'peck', 'walk']

fig, axes = plt.subplots(6, N, figsize=(20, 13))
for row, cls in enumerate(classes):
    files = []
    for split in ['train', 'val', 'test']:
        files += glob.glob(f'{OUT}/{split}/{cls}/*.jpg')
    files = random.sample(files, min(N, len(files)))
    for j, f in enumerate(files[:N]):
        ax = axes[row * 2][j]
        ax.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB))
        ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(cls.upper(), fontsize=15, fontweight='bold', color=colors[cls])
        name = os.path.basename(f).split('.')[0]
        ax.set_title(name[:18], fontsize=7)
        ctx = f.replace(OUT, CTX)
        cax = axes[row * 2 + 1][j]
        if os.path.exists(ctx):
            cax.imshow(cv2.cvtColor(cv2.imread(ctx), cv2.COLOR_BGR2RGB))
            cax.set_title('context', fontsize=7, color='gray')
        else:
            cax.text(0.5, 0.5, 'no ctx', ha='center', va='center')
        cax.set_xticks([]); cax.set_yticks([])
axes[0][0].set_title('CROP (top) + FULL-FRAME CONTEXT (bottom)', loc='left', fontsize=9, color='gray')
plt.tight_layout()
os.makedirs(f'{BASE}/reports', exist_ok=True)
plt.savefig(f'{BASE}/reports/d3_label_check.png', dpi=100, bbox_inches='tight')
print('Padded grid with context saved to', f'{BASE}/reports/d3_label_check.png')
plt.show()

### STOP - APPROVAL GATE
**Do NOT continue to training yet.**

Paste the outputs of Parts A-G (quantitative motion table, your verdict on the videos, and the crop+context grid) back to the user. Training in Part H-J starts only after the behavior labels are approved.

## PART H - Model A: YOLOv8n Segmentation Training (Dataset1)
Train a lightweight segmentation model to locate chickens. Metrics: Precision, Recall, mAP50, mAP50-95.

In [ ]:
from ultralytics import YOLO

D1 = '/content/datasets/D1_Detection'
model = YOLO('yolov8n-seg.pt')
r = model.train(data=f'{D1}/data.yaml',
                epochs=50, imgsz=640, batch=16, device=0,
                project='/content/runs', name='seg',
                patience=10, seed=0)
print('Final train metrics:')
print(r.results_dict)

In [ ]:
from ultralytics import YOLO

best = YOLO('/content/runs/seg/weights/best.pt')
m = best.val(data=f'{D1}/data.yaml', split='test')
print('Test set  (box):  mAP50=%.4f  mAP50-95=%.4f  P=%.4f  R=%.4f' % (m.box.map50, m.box.map, m.box.mp, m.box.mr))
print('Test set  (seg):  mAP50=%.4f  mAP50-95=%.4f  P=%.4f  R=%.4f' % (m.seg.map50, m.seg.map, m.seg.mp, m.seg.mr))

## PART H2 - Model A RETRAIN: YOLOv8n Segmentation (fixed recipe)

**Why retrain:** the Part H run stopped at epoch 1 with mAP50~0.25 and Precision~0.02. The auto-AdamW learning rate was too high for a 93-image set, and the 9-image validation set made early-stopping pure noise.

**Fixes:**
- `optimizer='SGD'` + `lr0=0.01` (stable on tiny datasets)
- `epochs=180` (was 50), `patience=50` (was 10), `cache=True`
- **Merged validation set: val(9) + test(4) = 13 images** so metrics are not decided by a handful of images.

The original 4-image held-out test set is still evaluated separately afterward.

In [ ]:
import os, glob, shutil

D1 = '/content/datasets/D1_Detection'
V2 = '/content/datasets/D1_Detection_v2'
if os.path.exists(V2):
    shutil.rmtree(V2)

# copy train
for sub in ['train', 'valid']:
    for f in glob.glob(f'{D1}/{sub}/images/*'):
        d = f'{V2}/images/{os.path.basename(f)}'
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copy(f, d)
    for f in glob.glob(f'{D1}/{sub}/labels/*'):
        d = f'{V2}/labels/{os.path.basename(f)}'
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copy(f, d)

# merged validation = original valid(9) + test(4) = 13
for sub in ['valid', 'test']:
    for f in glob.glob(f'{D1}/{sub}/images/*'):
        d = f'{V2}/val/images/{os.path.basename(f)}'
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copy(f, d)
    for f in glob.glob(f'{D1}/{sub}/labels/*'):
        d = f'{V2}/val/labels/{os.path.basename(f)}'
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copy(f, d)

nt = len(glob.glob(f'{V2}/images/*.jpg'))
nv = len(glob.glob(f'{V2}/val/images/*.jpg'))
print(f'train images: {nt}, merged valid images: {nv}')

with open(f'{V2}/data.yaml', 'w') as f:
    f.write('train: ../D1_Detection_v2/images\n')
    f.write('val: ../D1_Detection_v2/val/images\n')
    f.write('nc: 1\n')
    f.write("names: ['chicken']\n")
print(open(f'{V2}/data.yaml').read())

In [ ]:
from ultralytics import YOLO

V2 = '/content/datasets/D1_Detection_v2'
model = YOLO('yolov8n-seg.pt')
r = model.train(data=f'{V2}/data.yaml',
                epochs=180, imgsz=640, batch=16, device=0,
                optimizer='SGD', lr0=0.01, cache=True,
                project='/content/runs', name='seg_v2',
                patience=50, seed=0)
print('Final train metrics (v2):')
print(r.results_dict)

In [ ]:
from ultralytics import YOLO

D1 = '/content/datasets/D1_Detection'
V2 = '/content/datasets/D1_Detection_v2'

best = YOLO('/content/runs/seg_v2/weights/best.pt')
print('Merged 13-image validation set:')
m = best.val(data=f'{V2}/data.yaml')
print('  (box): mAP50=%.4f mAP50-95=%.4f P=%.4f R=%.4f' % (m.box.map50, m.box.map, m.box.mp, m.box.mr))
print('  (seg): mAP50=%.4f mAP50-95=%.4f P=%.4f R=%.4f' % (m.seg.map50, m.seg.map, m.seg.mp, m.seg.mr))

print('Original 4-image held-out test set:')
m2 = best.val(data=f'{D1}/data.yaml', split='test')
print('  (box): mAP50=%.4f mAP50-95=%.4f P=%.4f R=%.4f' % (m2.box.map50, m2.box.map, m2.box.mp, m2.box.mr))
print('  (seg): mAP50=%.4f mAP50-95=%.4f P=%.4f R=%.4f' % (m2.seg.map50, m2.seg.map, m2.seg.mp, m2.seg.mr))

print()
print('DECISION: if merged-val mAP50 < 0.5, run PART H3 (yolov8s-seg). Otherwise keep seg_v2.')

## PART H3 - Model A ESCALATION: YOLOv8s Segmentation (only if H2 mAP50 < ~0.5)

Run ONLY if Part H2's merged-validation mAP50 was below ~0.5. Same fixed recipe, larger model (`yolov8s-seg`) for more capacity on the 93-image set.

In [ ]:
from ultralytics import YOLO

V2 = '/content/datasets/D1_Detection_v2'
model = YOLO('yolov8s-seg.pt')
r = model.train(data=f'{V2}/data.yaml',
                epochs=180, imgsz=640, batch=16, device=0,
                optimizer='SGD', lr0=0.01, cache=True,
                project='/content/runs', name='seg_v3',
                patience=50, seed=0)
print('Final train metrics (v3):')
print(r.results_dict)

In [ ]:
from ultralytics import YOLO

D1 = '/content/datasets/D1_Detection'
V2 = '/content/datasets/D1_Detection_v2'

best = YOLO('/content/runs/seg_v3/weights/best.pt')
print('Merged 13-image validation set (v3):')
m = best.val(data=f'{V2}/data.yaml')
print('  (box): mAP50=%.4f mAP50-95=%.4f P=%.4f R=%.4f' % (m.box.map50, m.box.map, m.box.mp, m.box.mr))
print('  (seg): mAP50=%.4f mAP50-95=%.4f P=%.4f R=%.4f' % (m.seg.map50, m.seg.map, m.seg.mp, m.seg.mr))

print('Original 4-image held-out test set (v3):')
m2 = best.val(data=f'{D1}/data.yaml', split='test')
print('  (box): mAP50=%.4f mAP50-95=%.4f P=%.4f R=%.4f' % (m2.box.map50, m2.box.map, m2.box.mp, m2.box.mr))
print('  (seg): mAP50=%.4f mAP50-95=%.4f P=%.4f R=%.4f' % (m2.seg.map50, m2.seg.map, m2.seg.mp, m2.seg.mr))

## PART I - Model C: Health Classifier (Dataset2)
Train a lightweight classifier (YOLOv8-cls) on Healthy/Unhealthy. Then evaluate with accuracy/precision/recall/F1 + confusion matrix on the held-out test split.

In [ ]:
from ultralytics import YOLO

CLS = '/content/health_cls'
model = YOLO('yolov8n-cls.pt')
r = model.train(data=CLS,
                epochs=40, imgsz=224, batch=32, device=0,
                project='/content/runs', name='health',
                patience=10, seed=0)
print('Final train metrics:')
print(r.results_dict)

In [ ]:
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import glob

best = YOLO('/content/runs/health/weights/best.pt')
y_true = []
y_pred = []
for f in glob.glob('/content/health_cls/test/*/*.jpg'):
    true = f.split('/')[-2]
    res = best.predict(f, imgsz=224, verbose=False)[0]
    y_true.append(true)
    y_pred.append(res.names[res.probs.top1])

print('Test set (held-out test split):')
print('Accuracy: %.4f' % accuracy_score(y_true, y_pred))
print()
print(classification_report(y_true, y_pred, digits=4))
print('Confusion matrix:')
print(confusion_matrix(y_true, y_pred))

## PART J - Model B: Behavior Classifier (Dataset3 crops)
Train a frame-level classifier on tracked chicken crops labeled stand/peck/walk. Evaluate on the held-out video (000011). Also measure inference speed (README requires it).

In [ ]:
from ultralytics import YOLO

OUT = '/content/behavior_cls'
model = YOLO('yolov8n-cls.pt')
r = model.train(data=OUT,
                epochs=30, imgsz=224, batch=32, device=0,
                project='/content/runs', name='behavior',
                patience=8, seed=0)
print('Final train metrics:')
print(r.results_dict)

In [ ]:
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import glob

best = YOLO('/content/runs/behavior/weights/best.pt')
y_true = []
y_pred = []
for f in glob.glob('/content/behavior_cls/test/*/*.jpg'):
    true = f.split('/')[-2]
    res = best.predict(f, imgsz=224, verbose=False)[0]
    y_true.append(true)
    y_pred.append(res.names[res.probs.top1])

print('Test set (held-out video 000011):')
print('Accuracy: %.4f' % accuracy_score(y_true, y_pred))
print()
print(classification_report(y_true, y_pred, digits=4))
print('Confusion matrix:')
print(confusion_matrix(y_true, y_pred))

In [ ]:
import time, cv2, glob
from ultralytics import YOLO

best = YOLO('/content/runs/behavior/weights/best.pt')
samples = glob.glob('/content/behavior_cls/test/*/*.jpg')[:50]
t0 = time.time()
for f in samples:
    best.predict(f, imgsz=224, verbose=False)
t1 = time.time()
ms = (t1 - t0) / len(samples) * 1000
print('Inference: %.1f ms/frame (%.1f FPS) at imgsz=224' % (ms, 1000 / ms))

## PART K - Save Models and Experiments to Drive
Saves to `/content/drive/MyDrive/PoultrySense/models/` and `/experiments/`. Existing files are NOT overwritten.

For the segmentation model, the **newest good run wins**: it prefers `seg_v3` (H3) if present, otherwise `seg_v2` (H2). The failed `seg` baseline is preserved under a `_v1_baseline` name.

In [ ]:
import shutil, os

BASE = '/content/drive/MyDrive/PoultrySense'
mdir = os.path.join(BASE, 'models')
os.makedirs(mdir, exist_ok=True)

# 1) Segmentation: prefer seg_v3 (H3) -> seg_v2 (H2) -> keep existing seg
seg_src = None
for cand in ['/content/runs/seg_v3/weights/best.pt',
             '/content/runs/seg_v2/weights/best.pt']:
    if os.path.exists(cand):
        seg_src = cand
        break

seg_dst = os.path.join(mdir, 'PoultrySense_YOLOv8_ChickenSeg.pt')
if seg_src:
    # preserve any existing baseline under a versioned name
    if os.path.exists(seg_dst):
        bak = os.path.join(mdir, 'PoultrySense_YOLOv8_ChickenSeg_v1_baseline.pt')
        if not os.path.exists(bak):
            shutil.copy(seg_dst, bak)
            print('preserved old seg as:', os.path.basename(bak))
    shutil.copy(seg_src, seg_dst)
    print('saved (seg):', os.path.basename(seg_dst), '<-', os.path.basename(seg_src))
else:
    print('no seg_v2/v3 model found - keeping existing ChickenSeg if present')

# 2) Health + Behavior
for src, name in {
    '/content/runs/health/weights/best.pt':   'PoultrySense_HealthClassifier.pt',
    '/content/runs/behavior/weights/best.pt': 'PoultrySense_BehaviorModel.pt',
}.items():
    dst = os.path.join(mdir, name)
    if not os.path.exists(src):
        print('skip (missing):', name)
        continue
    if os.path.exists(dst):
        print('exists, not overwritten:', name)
    else:
        shutil.copy(src, dst)
        print('saved:', name)

# 3) Experiment dirs
for rname in ['seg', 'seg_v2', 'seg_v3', 'health', 'behavior']:
    src = f'/content/runs/{rname}'
    dst = f'{BASE}/experiments/{rname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
        print('copied experiment dir:', dst)
    else:
        print('exists, not overwritten:', dst)

print()
print('Models in Drive:')
if os.path.exists(mdir):
    for f in sorted(os.listdir(mdir)):
        print(' ', f, round(os.path.getsize(os.path.join(mdir, f)) / 1e6, 1), 'MB')

## DONE
Paste the full notebook output back to the user for review and the next steps (tracking + integrated pipeline).